# SVM eval: recorded datasets against `svm-best`

Scores the published RBF-SVM (`training/models/svm-best/`) on the **recorded** CQT datasets only: `training`, `thinkpad`, `vivo`, `flow`, `thinkpad-2`, `flow-2`. No noise / RIR / DIR overlays.

Checkpoints: predictions are written every `CHUNK` clips under `svm-eval-results/cache/{dataset}.npz`. A finished dataset is skipped on rerun unless `FORCE_REEVAL` is true. Progress and per-dataset metrics land next to the cache (`progress.csv`, `summary.csv`, `metrics/`, `confusion/`, `perclass/`).

Run with nbconvert, not a live MCP execute of the eval cell.

## Config

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATASETS = ["thinkpad", "vivo", "flow", "thinkpad-2", "flow-2", "training"]
CHUNK = 200
FORCE_REEVAL = False


def find_training_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "features").is_dir() and (candidate / "models").is_dir():
            return candidate
        nested = candidate / "training"
        if (nested / "features").is_dir() and (nested / "models").is_dir():
            return nested
    raise FileNotFoundError(f"Could not locate training root from {start}")


HERE = Path.cwd().resolve()
TRAINING_ROOT = find_training_root(HERE)
FEATURES_DIR = TRAINING_ROOT / "features"
MODEL_DIR = TRAINING_ROOT / "models" / "svm-best"
NOTEBOOK_DIR = HERE if (HERE / "svm-eval-recorded.ipynb").exists() else (TRAINING_ROOT / "notebooks" / "svm")
OUT_DIR = NOTEBOOK_DIR / "svm-eval-results"

for name in DATASETS:
    assert (FEATURES_DIR / f"{name}.npz").is_file(), FEATURES_DIR / f"{name}.npz"
for p in (MODEL_DIR / "model.pkl", MODEL_DIR / "scaler.pkl", MODEL_DIR / "label_encoder.pkl"):
    assert p.is_file(), p

for sub in ("cache", "metrics", "confusion", "perclass"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

cfg = pd.DataFrame(
    {
        "key": ["datasets", "chunk", "force_reeval", "features", "model", "out"],
        "value": [
            ", ".join(DATASETS),
            CHUNK,
            FORCE_REEVAL,
            str(FEATURES_DIR),
            str(MODEL_DIR),
            str(OUT_DIR),
        ],
    }
)
display(cfg)

,key,value
0,datasets,"thinkpad, vivo, flow, thinkpad-2, flow-2, trai..."
1,chunk,200
2,force_reeval,False
3,features,/home/seya/code/chord-detection/training/features
4,model,/home/seya/code/chord-detection/training/model...
5,out,/home/seya/code/chord-detection/training/noteb...


## Load trained SVM

In [2]:
import warnings

import joblib
import pandas as pd
from IPython.display import display
from sklearn.exceptions import InconsistentVersionWarning

warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

svm = joblib.load(MODEL_DIR / "model.pkl")
scaler = joblib.load(MODEL_DIR / "scaler.pkl")
encoder = joblib.load(MODEL_DIR / "label_encoder.pkl")
CLASS_NAMES = list(encoder.classes_)

display(
    pd.DataFrame(
        [
            {
                "estimator": type(svm).__name__,
                "kernel": getattr(svm, "kernel", None),
                "C": getattr(svm, "C", None),
                "gamma": getattr(svm, "gamma", None),
                "n_support": int(getattr(svm, "n_support_", [0]).sum()),
                "n_classes": len(CLASS_NAMES),
            }
        ]
    )
)

,estimator,kernel,C,gamma,n_support,n_classes
0,SVC,rbf,1000,0.00001,4143,36


## Evaluate recorded datasets

In [3]:
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

PROGRESS_PATH = OUT_DIR / "progress.csv"
SUMMARY_PATH = OUT_DIR / "summary.csv"
CACHE_DIR = OUT_DIR / "cache"


def cache_path(name: str) -> Path:
    return CACHE_DIR / f"{name}.npz"


def save_cache(name: str, y_true, y_pred, next_index: int) -> None:
    np.savez_compressed(
        cache_path(name),
        y_true=np.asarray(y_true),
        y_pred=np.asarray(y_pred),
        next_index=np.int64(next_index),
        n=np.int64(len(y_true)),
    )


def load_cache(name: str, n: int):
    p = cache_path(name)
    if FORCE_REEVAL or not p.is_file():
        return None
    with np.load(p) as z:
        if int(z["n"]) != n:
            return None
        return {
            "y_true": z["y_true"].copy(),
            "y_pred": z["y_pred"].copy(),
            "next_index": int(z["next_index"]),
        }


def write_tables(name: str, y_true, y_pred) -> dict:
    acc = float(accuracy_score(y_true, y_pred))
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(len(CLASS_NAMES)), average="macro", zero_division=0
    )
    wp, wr, wf1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(len(CLASS_NAMES)), average="weighted", zero_division=0
    )
    report = classification_report(
        y_true, y_pred, labels=np.arange(len(CLASS_NAMES)), target_names=CLASS_NAMES, zero_division=0, output_dict=True
    )
    report_df = pd.DataFrame(report).T
    report_df.to_csv(OUT_DIR / "metrics" / f"{name}.csv")
    per_class = report_df.loc[CLASS_NAMES, ["precision", "recall", "f1-score", "support"]]
    per_class.to_csv(OUT_DIR / "perclass" / f"{name}.csv")
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASS_NAMES)))
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        OUT_DIR / "confusion" / f"{name}.csv"
    )
    return {
        "dataset": name,
        "n_samples": int(len(y_true)),
        "n_classes": len(CLASS_NAMES),
        "accuracy": acc,
        "macro_precision": float(p),
        "macro_recall": float(r),
        "macro_f1": float(f1),
        "weighted_precision": float(wp),
        "weighted_recall": float(wr),
        "weighted_f1": float(wf1),
    }


def append_progress(row: dict) -> None:
    df = pd.DataFrame([row])
    if PROGRESS_PATH.is_file():
        prev = pd.read_csv(PROGRESS_PATH)
        prev = prev[prev["dataset"] != row["dataset"]]
        df = pd.concat([prev, df], ignore_index=True)
    df.to_csv(PROGRESS_PATH, index=False)
    df.to_csv(SUMMARY_PATH, index=False)


rows = []
for name in DATASETS:
    feat_path = FEATURES_DIR / f"{name}.npz"
    with np.load(feat_path, allow_pickle=True) as data:
        features = data["features"]
        labels = np.asarray(data["labels"]).astype(str)
    X = features.reshape(features.shape[0], -1)
    y_true = encoder.transform(labels)
    n = len(y_true)
    cached = load_cache(name, n)
    if cached is not None and cached["next_index"] >= n:
        y_pred = cached["y_pred"]
        status = "cache_hit"
        elapsed = 0.0
        print(f"{name}: cache hit n={n}", flush=True)
    else:
        if cached is not None:
            y_pred = cached["y_pred"]
            start = cached["next_index"]
            status = "resumed"
            print(f"{name}: resume at {start}/{n}", flush=True)
        else:
            y_pred = np.full(n, -1, dtype=np.int64)
            start = 0
            status = "fresh"
            print(f"{name}: start n={n}", flush=True)
        t0 = time.perf_counter()
        for i in range(start, n, CHUNK):
            j = min(i + CHUNK, n)
            y_pred[i:j] = svm.predict(scaler.transform(X[i:j]))
            save_cache(name, y_true, y_pred, j)
            print(f"  {name} {j}/{n}", flush=True)
        elapsed = time.perf_counter() - t0
        status = "done" if status == "fresh" else "resumed_done"
    metrics = write_tables(name, y_true, y_pred)
    metrics.update(
        {
            "status": status,
            "predict_s": round(elapsed, 2),
            "finished_at": datetime.now(timezone.utc).isoformat(),
            "cache_path": str(cache_path(name)),
        }
    )
    append_progress(metrics)
    rows.append(metrics)
    print(f"{name}: acc={metrics['accuracy']:.4f} status={status} {elapsed:.1f}s", flush=True)

display(pd.DataFrame(rows))

thinkpad: start n=1440


  thinkpad 200/1440


  thinkpad 400/1440


  thinkpad 600/1440


  thinkpad 800/1440


...

  training 6800/7200


  training 7000/7200


  training 7200/7200


training: acc=1.0000 status=done 1922.8s


,dataset,n_samples,n_classes,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,status,predict_s,finished_at,cache_path
0,thinkpad,1440,36,0.861806,0.935433,0.861806,0.873961,0.935433,0.861806,0.873961,done,613.01,2026-08-15T13:29:15.522120+00:00,/home/seya/code/chord-detection/training/noteb...
1,vivo,1440,36,0.977778,0.983888,0.977778,0.976866,0.983888,0.977778,0.976866,done,475.08,2026-08-15T13:37:11.832889+00:00,/home/seya/code/chord-detection/training/noteb...
2,flow,1440,36,0.994444,0.994976,0.994444,0.994522,0.994976,0.994444,0.994522,done,479.53,2026-08-15T13:45:12.659869+00:00,/home/seya/code/chord-detection/training/noteb...
3,thinkpad-2,1440,36,0.571528,0.762374,0.571528,0.573338,0.762374,0.571528,0.573338,done,509.14,2026-08-15T13:53:43.119863+00:00,/home/seya/code/chord-detection/training/noteb...
4,flow-2,1440,36,0.793750,0.826583,0.793750,0.781224,0.826583,0.793750,0.781224,done,368.09,2026-08-15T13:59:52.514958+00:00,/home/seya/code/chord-detection/training/noteb...
5,training,7200,36,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,done,1922.82,2026-08-15T14:31:57.311516+00:00,/home/seya/code/chord-detection/training/noteb...


## Summary

In [4]:
from IPython.display import display
import pandas as pd

summary = pd.read_csv(OUT_DIR / "summary.csv")
display(summary[["dataset", "n_samples", "accuracy", "macro_precision", "macro_recall", "macro_f1", "status", "predict_s"]])

,dataset,n_samples,accuracy,macro_precision,macro_recall,macro_f1,status,predict_s
0,thinkpad,1440,0.861806,0.935433,0.861806,0.873961,done,613.01
1,vivo,1440,0.977778,0.983888,0.977778,0.976866,done,475.08
2,flow,1440,0.994444,0.994976,0.994444,0.994522,done,479.53
3,thinkpad-2,1440,0.571528,0.762374,0.571528,0.573338,done,509.14
4,flow-2,1440,0.793750,0.826583,0.793750,0.781224,done,368.09
5,training,7200,1.000000,1.000000,1.000000,1.000000,done,1922.82
